# TP 1 — Premiers pas en Deep Learning et Big Data

**M1 · Deep Learning & Big Data · Séance 1 · Durée : 35 min en séance (+ fin à la maison)**

Au programme :

| Partie | Contenu | Priorité |
|---|---|---|
| 0 | Préparer l'environnement | ⭐ obligatoire |
| 1 | Classer vos propres photos avec un modèle déjà entraîné | ⭐ obligatoire |
| 2 | Entraîner votre premier réseau de neurones | ⭐ obligatoire |
| 3 | Premier contact avec Spark : Pandas contre PySpark | à finir à la maison si besoin |
| 4 | Créer votre compte Databricks Free Edition | avant la séance 2 |
| 5 | Choisir votre projet fil rouge | à rendre en fin de séance |

**Mode d'emploi**
- Exécutez les cellules dans l'ordre avec `Maj + Entrée`.
- Les cellules marquées `# TODO` sont à compléter : remplacez les `____` par le bon code.
- Les questions ❓ appellent une réponse courte, à écrire dans la cellule texte juste en dessous.
- Aucune formule mathématique n'est nécessaire : lisez les explications, testez, observez.

---
## Partie 0 — Préparer l'environnement

1. Menu **Exécution › Modifier le type d'exécution** → choisissez un **GPU** (par exemple T4) puis **Enregistrer**.
2. Exécutez la cellule ci-dessous. Elle doit afficher au moins un GPU.

> 💡 Un GPU est une carte graphique : elle sait faire des milliers de petits calculs en même temps, exactement ce dont un réseau de neurones a besoin.

In [ ]:
import tensorflow as tf
import keras

print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU détecté :", gpus if gpus else "aucun (le TP fonctionne quand même, en plus lent)")

---
## Partie 1 — Classer des images sans écrire une seule règle

On utilise un modèle **déjà entraîné** par Google sur ImageNet, une base de plus d'un million d'images réparties en **1 000 catégories** (en anglais : *tabby cat*, *sports car*, *espresso*…).

Personne n'a écrit de règle du type « si oreilles pointues alors chat » : le modèle a appris tout seul à partir d'exemples étiquetés. C'est le changement de point de vue vu en cours.

La bibliothèque **Hugging Face Transformers** permet de le charger en une ligne avec `pipeline`.

In [ ]:
from transformers import pipeline

# Téléchargement du modèle (environ 350 Mo, une seule fois)
classifieur = pipeline("image-classification", model="google/vit-base-patch16-224")
print("Modèle chargé ✅")

In [ ]:
# Test sur une image publique de la documentation Hugging Face
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"

from IPython.display import Image as AfficherImage, display
display(AfficherImage(url=url, width=300))

resultats = classifieur(url)
for r in resultats:
    print(f"{r['label']:<45} {r['score']:.1%}")

### 🛠️ Exercice 1.1 — Vos propres photos

Importez 2 ou 3 photos depuis votre ordinateur ou votre téléphone (objets, animaux, plats, véhicules…) et faites-les classer par le modèle.

> 💡 Le classifieur prend en entrée le **nom du fichier** et renvoie une liste de dictionnaires avec une étiquette (`label`) et un score de confiance (`score`).

In [ ]:
from google.colab import files

fichiers = files.upload()  # une fenêtre s'ouvre pour choisir vos images

for nom in fichiers:
    display(AfficherImage(filename=nom, width=250))
    resultats = ____  # TODO : appelez le classifieur sur l'image `nom`
    for r in resultats[:3]:
        print(f"   {r['label']:<45} {r['score']:.1%}")
    print("-" * 60)

### ❓ Question 1.2

Trouvez une image sur laquelle le modèle **se trompe** ou hésite (score faible). À votre avis, pourquoi ?

*Votre réponse :*

---
## Partie 2 — Entraîner votre premier réseau de neurones

Cette fois, c'est **vous** qui entraînez un modèle, à partir de zéro.

**Les données : Fashion-MNIST.** 70 000 petites images en niveaux de gris (28 × 28 pixels) de vêtements, réparties en 10 catégories : t-shirt, pantalon, pull, robe, manteau, sandale, chemise, basket, sac, bottine.
- 60 000 images servent à **entraîner** le modèle ;
- 10 000 images, jamais vues pendant l'entraînement, servent à le **tester**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from keras import layers

(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

classes = ["T-shirt", "Pantalon", "Pull", "Robe", "Manteau",
           "Sandale", "Chemise", "Basket", "Sac", "Bottine"]

print("Images d'entraînement :", x_train.shape)   # (nombre d'images, hauteur, largeur)
print("Images de test        :", x_test.shape)
print("Une étiquette         :", y_train[0], "→", classes[y_train[0]])

In [ ]:
# Aperçu des 15 premières images
plt.figure(figsize=(12, 3))
for i in range(15):
    plt.subplot(2, 8, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(classes[y_train[i]], fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

### Préparer les données

Chaque pixel vaut entre 0 (noir) et 255 (blanc). Les réseaux de neurones apprennent beaucoup mieux avec des valeurs **entre 0 et 1** : on divise donc tout par 255.

In [ ]:
x_train = x_train / 255.0
x_test = x_test / 255.0
print("Valeur min :", x_train.min(), "· valeur max :", x_train.max())

### 🛠️ Exercice 2.1 — Construire le réseau

On empile des couches, comme des briques de Lego :

1. `Flatten` : « déplie » l'image 28 × 28 en une simple liste de 784 pixels ;
2. `Dense` cachée : une couche de neurones qui cherche des motifs (prenez **128 neurones**) ;
3. `Dense` de sortie : **un neurone par catégorie**. L'activation `softmax` transforme les sorties en probabilités qui totalisent 100 %.

Complétez les `____`.

In [ ]:
modele = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(____, activation="relu"),     # TODO : nombre de neurones de la couche cachée
    layers.Dense(____, activation="softmax"),  # TODO : combien de catégories à prédire ?
])

modele.summary()

> 💡 Dans le résumé, la colonne **Param #** compte les **poids** que le réseau va ajuster pendant l'apprentissage. Plus de 100 000 réglages à trouver… heureusement, c'est Keras qui s'en charge.

### Préparer l'apprentissage

On indique à Keras trois choses, sans avoir besoin des formules :
- **`optimizer="adam"`** : la méthode pour ajuster les poids (le randonneur qui descend la colline dans le brouillard) ;
- **`loss`** : la façon de mesurer l'erreur (la « hauteur » à faire baisser) ;
- **`metrics=["accuracy"]`** : ce qu'on affiche pour nous, humains : le pourcentage de bonnes réponses.

In [ ]:
modele.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

### 🛠️ Exercice 2.2 — Lancer l'entraînement

Une **époque** (*epoch*) = le modèle a vu une fois toutes les images d'entraînement. Lancez **5 époques**.

On met de côté 10 % des images d'entraînement (`validation_split=0.1`) pour surveiller, à chaque époque, comment le modèle se débrouille sur des images qu'il n'apprend pas.

In [ ]:
historique = modele.fit(
    x_train, y_train,
    epochs=____,            # TODO : nombre d'époques
    validation_split=0.1,
)

In [ ]:
# Courbes d'apprentissage
plt.figure(figsize=(7, 4))
plt.plot(historique.history["accuracy"], marker="o", label="Entraînement")
plt.plot(historique.history["val_accuracy"], marker="o", label="Validation")
plt.xlabel("Époque")
plt.ylabel("Bonnes réponses")
plt.title("Le modèle apprend")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Le vrai test : des images jamais vues

In [ ]:
perte, precision = modele.evaluate(x_test, y_test, verbose=0)
print(f"Bonnes réponses sur les 10 000 images de test : {precision:.1%}")

In [ ]:
# Prédictions sur 12 images de test (en rouge : les erreurs)
probas = modele.predict(x_test[:12], verbose=0)
predictions = probas.argmax(axis=1)

plt.figure(figsize=(12, 4))
for i in range(12):
    plt.subplot(2, 6, i + 1)
    plt.imshow(x_test[i], cmap="gray")
    ok = predictions[i] == y_test[i]
    plt.title(f"{classes[predictions[i]]}\n({probas[i].max():.0%})",
              color="green" if ok else "red", fontsize=9)
    plt.axis("off")
plt.suptitle("Prédiction du modèle (confiance) — vert : correct, rouge : erreur")
plt.tight_layout()
plt.show()

### 🛠️ Exercice 2.3 — Expérimentez

Modifiez **une seule chose à la fois** dans les cellules précédentes, réexécutez depuis l'exercice 2.1, puis notez le score de test obtenu.

| Expérience | Modification | Score de test |
|---|---|---|
| Référence | 128 neurones, 5 époques | |
| A | 16 neurones au lieu de 128 | |
| B | 512 neurones au lieu de 128 | |
| C | 1 époque au lieu de 5 | |
| D | 20 époques au lieu de 5 | |
| E | Ajouter une 2e couche `Dense(128, activation="relu")` | |

### ❓ Question 2.4

Dans l'expérience D, comparez les courbes « Entraînement » et « Validation ». Que remarquez-vous ? Comment l'expliquez-vous ?

*Votre réponse :*

---
## Partie 3 — Premier contact avec Spark

Apache Spark répartit les données et les calculs sur plusieurs machines (un **cluster**). Ici, nous n'avons qu'une machine Colab : Spark va simuler un cluster en utilisant les différents cœurs du processeur. C'est suffisant pour apprendre la syntaxe, qui sera **exactement la même** sur un vrai cluster.

On va comparer **Pandas** (une machine, tout en mémoire) et **PySpark** (Spark piloté en Python) sur un jeu de 5 millions de ventes fictives.

In [ ]:
# Installation de PySpark (et de Java si nécessaire)
import shutil
!pip install -q pyspark
if shutil.which("java") is None:
    !apt-get install -y -qq openjdk-17-jre-headless > /dev/null
print("Java :", shutil.which("java"))

In [ ]:
# Génération d'un jeu de données de ventes fictives (5 millions de lignes)
import pandas as pd

rng = np.random.default_rng(42)
n = 5_000_000
villes = ["Paris", "Lyon", "Marseille", "Toulouse", "Lille", "Rouen", "Nantes", "Bordeaux"]
poids_villes = [0.30, 0.14, 0.12, 0.10, 0.09, 0.07, 0.09, 0.09]   # Paris pèse plus lourd
prix = {"Ordinateur": 900, "Téléphone": 600, "Écran": 220, "Casque": 90, "Clavier": 45, "Souris": 25}
produits = list(prix)

produit = rng.choice(produits, n, p=[0.10, 0.20, 0.10, 0.20, 0.20, 0.20])
quantite = rng.integers(1, 4, n)
prix_unitaire = np.array([prix[p] for p in produits])[
    pd.Categorical(produit, categories=produits).codes]

ventes_pd = pd.DataFrame({
    "id_vente": np.arange(n),
    "ville": rng.choice(villes, n, p=poids_villes),
    "produit": produit,
    "quantite": quantite,
    # prix unitaire × quantité, avec ±20 % de variation (promotions, gammes…)
    "montant": (prix_unitaire * quantite * rng.uniform(0.8, 1.2, n)).round(2),
})
ventes_pd.to_parquet("ventes.parquet", index=False)

print(f"{len(ventes_pd):,} lignes générées".replace(",", " "))
ventes_pd.head()

> 💡 **Parquet** est un format de fichier « en colonnes » très utilisé en Big Data : il est compressé et permet de ne lire que les colonnes utiles.

### Avec Pandas : chiffre d'affaires par ville

In [ ]:
%%time
ventes_pd.groupby("ville")["montant"].sum().sort_values(ascending=False)

### Avec PySpark : la même question

On crée d'abord une **SparkSession**, le point d'entrée vers Spark. Sur Databricks, elle existe déjà sous le nom `spark`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("TP1").getOrCreate()

ventes = spark.read.parquet("ventes.parquet")
ventes.printSchema()
ventes.show(5)

### L'évaluation paresseuse (*lazy evaluation*)

Exécutez la cellule suivante : elle est **instantanée**. Spark n'a encore rien calculé ! Il a seulement noté **le plan** de ce qu'il faudra faire.

In [ ]:
%%time
requete = (ventes
           .groupBy("ville")
           .agg(F.round(F.sum("montant")).cast("long").alias("chiffre_affaires"))
           .orderBy(F.desc("chiffre_affaires")))
print(type(requete))

Le calcul ne démarre que lorsqu'on demande un **résultat** (`show()`, `count()`, `collect()`, écrire un fichier…). Cela permet à Spark d'optimiser tout le plan avant de lancer le travail sur le cluster.

In [ ]:
%%time
requete.show()

### 🛠️ Exercice 3.1 — Panier moyen par produit

Calculez le **montant moyen** d'une vente pour chaque produit, du plus cher au moins cher.

> 💡 Même structure que la requête ci-dessus, en remplaçant `ville` par `produit` et `F.sum` par `F.avg`.

In [ ]:
(ventes
 .groupBy(____)                                       # TODO : regrouper par produit
 .agg(____("montant").alias("panier_moyen"))          # TODO : fonction de moyenne
 .orderBy(F.desc("panier_moyen"))
 .show())

### 🛠️ Exercice 3.2 — Filtrer

Combien de ventes **à Rouen** ont un montant **supérieur à 1 000 €** ?

> 💡 `filter` garde les lignes qui respectent une condition ; `count()` compte les lignes. En Spark, on combine deux conditions avec `&` et on met chaque condition entre parenthèses.

In [ ]:
nb = (ventes
      .filter((F.col("ville") == ____) & (F.col("montant") > ____))   # TODO
      .count())
print("Ventes à Rouen de plus de 1 000 € :", nb)

### 🛠️ Exercice 3.3 (bonus) — Et si on préfère le SQL ?

Spark comprend aussi le SQL. Écrivez une requête qui donne, pour chaque ville, le **nombre de ventes** et la **quantité totale** vendue.

In [ ]:
ventes.createOrReplaceTempView("ventes")

spark.sql('''
    SELECT ville,
           ____ AS nb_ventes,        -- TODO : compter les lignes
           ____ AS quantite_totale   -- TODO : somme des quantités
    FROM ventes
    GROUP BY ville
    ORDER BY nb_ventes DESC
''').show()

### ❓ Question 3.4

Sur Colab, Spark n'est probablement **pas plus rapide** que Pandas (il est même souvent plus lent). Pourquoi l'utiliser alors ?

*Votre réponse :*

In [ ]:
# Libérer les ressources Spark à la fin de la partie 3
spark.stop()

---
## Partie 4 — Créer votre compte Databricks Free Edition (avant la séance 2)

En séances 2 et 3, nous utiliserons **Databricks Free Edition**, une version gratuite de la plateforme Databricks pensée pour les étudiants et les enseignants. Elle remplace l'ancienne *Community Edition*, retirée en 2025.

**Étapes :**
1. Rendez-vous sur la page d'inscription à Databricks Free Edition (lien depuis la documentation officielle : <https://docs.databricks.com/aws/fr/getting-started/free-edition>).
2. Inscrivez-vous avec votre adresse e-mail de l'école (pas de carte bancaire demandée).
3. Une fois dans votre espace de travail : **New › Notebook**.
4. Collez le code ci-dessous dans une cellule de **votre notebook Databricks** (pas ici !) et exécutez-le.

```python
# Dans Databricks, la variable `spark` existe déjà : pas besoin de SparkSession.builder
df = spark.range(1, 11).withColumnRenamed("id", "nombre")
display(df)
```

✅ Vous devez voir un tableau des nombres de 1 à 10. Si c'est le cas, vous êtes prêt·e pour la séance 2.

- [ ] Compte Databricks créé
- [ ] Premier notebook exécuté

---
## Partie 5 — Votre projet fil rouge

Par groupes de 2 ou 3, choisissez un jeu de données **d'images ou de texte** et une question à laquelle un modèle pourrait répondre. Vous le préparerez avec Spark (séances 2–3), puis vous entraînerez ou adapterez un modèle (séances 4–6).

**Où chercher :** [Kaggle Datasets](https://www.kaggle.com/datasets) · [Hugging Face Datasets](https://huggingface.co/datasets)

**Exemples de sujets :** classer des avis clients en positif ou négatif · reconnaître des types de déchets sur des photos · détecter des e-mails de phishing · classer des articles de presse par thème.

Complétez la fiche ci-dessous (double-cliquez pour éditer) :

| | Votre réponse |
|---|---|
| Membres du groupe | |
| Titre du projet | |
| Lien vers le dataset | |
| Type de données (images / texte) | |
| Taille (nombre d'exemples, Go) | |
| Question à laquelle le modèle répondra | |
| Pourquoi Spark est utile pour préparer ces données | |

---
## 🎯 Pour vérifier que vous avez compris

1. Dans la partie 1, qui a écrit les règles qui permettent de reconnaître un chat ?
2. Que mesure le score sur les images de **test**, et pourquoi est-il plus fiable que le score d'entraînement ?
3. Pourquoi Spark n'exécute-t-il rien tant qu'on ne lui demande pas de résultat ?

*Vos réponses :*